# 한국어 영화 리뷰 감정분석 모델 파인튜닝

이번 실습에서는 Hugging Face의 사전학습 모델을 가져와서  
한국어 영화 리뷰 감정분석 모델로 파인튜닝해보겠습니다.

이번 실습의 목표는 다음과 같습니다.

- 기본 한국어 BERT 모델을 불러온다.
- NSMC 영화 리뷰 데이터셋을 불러온다.
- 영화 리뷰를 긍정/부정으로 분류하도록 모델을 추가 학습한다.
- 파인튜닝 전과 후의 결과를 비교한다.

이번 실습에서 사용할 구성은 다음과 같습니다.

```text
기본 모델: klue/bert-base
데이터셋: NSMC
분류 라벨: 부정, 긍정
분류 방식: 단일 라벨 이진분류


---

```markdown
## 0. 라이브러리 설치

먼저 실습에 필요한 라이브러리를 설치합니다.

- `transformers`: Hugging Face 모델과 Trainer를 사용하기 위한 라이브러리
- `datasets`: Hugging Face 데이터셋을 불러오기 위한 라이브러리
- `evaluate`: 정확도 같은 평가 지표를 계산하기 위한 라이브러리
- `accelerate`: 학습 실행을 도와주는 라이브러리

In [ ]:
!pip install -q -U "datasets<4.0.0" transformers evaluate accelerate

## 1. 라이브러리 불러오기

이번 실습에서 사용할 라이브러리를 불러옵니다.

`torch`는 딥러닝 모델 실행에 사용하고,  
`datasets`는 NSMC 데이터셋을 불러오는 데 사용합니다.

In [ ]:
import torch
import numpy as np
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    pipeline
)
import evaluate

print("GPU 사용 가능 여부:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU 이름:", torch.cuda.get_device_name(0))

GPU 사용 가능 여부: True
GPU 이름: Tesla T4


## 2. 데이터셋 불러오기

이번 실습에서는 NSMC 데이터셋을 사용합니다.

NSMC는 네이버 영화 리뷰 문장과 긍정/부정 라벨로 구성된 한국어 감정분석 데이터셋입니다.

라벨의 의미는 다음과 같습니다.

```text
0 = 부정
1 = 긍정

In [ ]:
from datasets import load_dataset

dataset = load_dataset("e9t/nsmc", trust_remote_code=True)

dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'document', 'label'],
        num_rows: 150000
    })
    test: Dataset({
        features: ['id', 'document', 'label'],
        num_rows: 50000
    })
})

## 3. 데이터 확인하기

데이터셋 안에 어떤 값이 들어있는지 확인해봅시다.

NSMC 데이터셋에는 보통 다음 컬럼이 있습니다.

- `id`: 리뷰 ID
- `document`: 영화 리뷰 문장
- `label`: 감정 라벨

In [ ]:
dataset["train"][0]

{'id': '9976970', 'document': '아 더빙.. 진짜 짜증나네요 목소리', 'label': 0}

In [ ]:
for i in range(20):
    print("리뷰:", dataset["train"][i]["document"])
    print("라벨:", dataset["train"][i]["label"])
    print()

리뷰: 아 더빙.. 진짜 짜증나네요 목소리
라벨: 0

리뷰: 흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나
라벨: 1

리뷰: 너무재밓었다그래서보는것을추천한다
라벨: 0

리뷰: 교도소 이야기구먼 ..솔직히 재미는 없다..평점 조정
라벨: 0

리뷰: 사이몬페그의 익살스런 연기가 돋보였던 영화!스파이더맨에서 늙어보이기만 했던 커스틴 던스트가 너무나도 이뻐보였다
라벨: 1

리뷰: 막 걸음마 뗀 3세부터 초등학교 1학년생인 8살용영화.ㅋㅋㅋ...별반개도 아까움.
라벨: 0

리뷰: 원작의 긴장감을 제대로 살려내지못했다.
라벨: 0

리뷰: 별 반개도 아깝다 욕나온다 이응경 길용우 연기생활이몇년인지..정말 발로해도 그것보단 낫겟다 납치.감금만반복반복..이드라마는 가족도없다 연기못하는사람만모엿네
라벨: 0

리뷰: 액션이 없는데도 재미 있는 몇안되는 영화
라벨: 1

리뷰: 왜케 평점이 낮은건데? 꽤 볼만한데.. 헐리우드식 화려함에만 너무 길들여져 있나?
라벨: 1

리뷰: 걍인피니트가짱이다.진짜짱이다♥
라벨: 1

리뷰: 볼때마다 눈물나서 죽겠다90년대의 향수자극!!허진호는 감성절제멜로의 달인이다~
라벨: 1

리뷰: 울면서 손들고 횡단보도 건널때 뛰쳐나올뻔 이범수 연기 드럽게못해
라벨: 0

리뷰: 담백하고 깔끔해서 좋다. 신문기사로만 보다 보면 자꾸 잊어버린다. 그들도 사람이었다는 것을.
라벨: 1

리뷰: 취향은 존중한다지만 진짜 내생에 극장에서 본 영화중 가장 노잼 노감동임 스토리도 어거지고 감동도 어거지
라벨: 0

리뷰: ㄱ냥 매번 긴장되고 재밋음ㅠㅠ
라벨: 1

리뷰: 참 사람들 웃긴게 바스코가 이기면 락스코라고 까고바비가 이기면 아이돌이라고 깐다.그냥 까고싶어서 안달난것처럼 보인다
라벨: 1

리뷰: 굿바이 레닌 표절인것은 이해하는데 왜 뒤로 갈수록 재미없어지냐
라벨: 0

리뷰: 이건 정말 깨알 캐스팅과 질퍽하지않은 산뜻한 내용구성이 잘 버무러진 깨알일드!!♥
라벨: 1

리뷰: 약탈자를 위한 변명, 이라. 저놈들은 착한놈들 절대 

## 4. 실습용 데이터 줄이기

전체 데이터셋을 모두 사용하면 학습 시간이 길어질 수 있습니다.

이번 실습에서는 빠르게 결과를 확인하기 위해  
학습 데이터 2,000개, 테스트 데이터 500개만 사용하겠습니다.

더 좋은 성능을 원하면 데이터 개수를 늘릴 수 있습니다.

In [ ]:
train_dataset = dataset["train"].shuffle(seed=42).select(range(3000))
test_dataset = dataset["test"].shuffle(seed=42).select(range(1000))

print(train_dataset)
print(test_dataset)

Dataset({
    features: ['id', 'document', 'label'],
    num_rows: 3000
})
Dataset({
    features: ['id', 'document', 'label'],
    num_rows: 1000
})


## 5. 기본 모델과 토크나이저 불러오기

이번 실습에서는 `klue/bert-base` 모델을 사용합니다.

이 모델은 한국어를 이해하도록 사전학습된 BERT 모델입니다.  
하지만 처음부터 영화 리뷰 감정분석용으로 학습된 것은 아닙니다.

그래서 영화 리뷰 데이터를 이용해 긍정/부정을 분류하도록 추가 학습합니다.

이 과정을 파인튜닝이라고 합니다.

In [ ]:
model_name = "klue/bert-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

## 6. 파인튜닝 전 모델 만들어보기

`AutoModelForSequenceClassification`은 문장 분류용 모델을 만들 때 사용합니다.

여기서는 긍정/부정 2개 라벨을 분류해야 하므로 `num_labels=2`로 설정합니다.

주의할 점은, 이 시점의 분류기는 아직 NSMC 데이터로 학습되지 않았습니다.  
따라서 파인튜닝 전 예측 결과는 이상하거나 거의 랜덤처럼 나올 수 있습니다.

In [ ]:
model_before = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: klue/bert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
print("모델 레포 이름:", model_before.config.name_or_path)
print("모델 클래스:", model_before.__class__.__name__)
print("모델 타입:", model_before.config.model_type)
print("아키텍처:", model_before.config.architectures)

모델 레포 이름: klue/bert-base
모델 클래스: BertForSequenceClassification
모델 타입: bert
아키텍처: ['BertForMaskedLM']


## 7. 파인튜닝 전 예측 확인하기

파인튜닝하기 전에 모델이 어떤 결과를 내는지 먼저 확인해봅시다.

아직 감정분석 데이터로 학습하지 않았기 때문에 결과가 정확하지 않을 수 있습니다

In [ ]:
before_classifier = pipeline(
    "text-classification",
    model=model_before,
    tokenizer=tokenizer,
    device=0 if torch.cuda.is_available() else -1
)

test_texts = [
    "정말 재미있고 감동적인 영화였다.",
    "시간이 아까울 정도로 지루했다.",
    "배우들의 연기가 훌륭했다.",
    "스토리가 너무 엉성하고 재미없었다."
]

for text in test_texts:
    result = before_classifier(text)
    print("입력:", text)
    print("파인튜닝 전 결과:", result)
    print()

입력: 정말 재미있고 감동적인 영화였다.
파인튜닝 전 결과: [{'label': 'LABEL_0', 'score': 0.5856019854545593}]

입력: 시간이 아까울 정도로 지루했다.
파인튜닝 전 결과: [{'label': 'LABEL_0', 'score': 0.614448606967926}]

입력: 배우들의 연기가 훌륭했다.
파인튜닝 전 결과: [{'label': 'LABEL_0', 'score': 0.6459136009216309}]

입력: 스토리가 너무 엉성하고 재미없었다.
파인튜닝 전 결과: [{'label': 'LABEL_0', 'score': 0.6412321329116821}]



## 8. 문장을 토큰으로 변환하기

BERT 모델은 문장을 그대로 입력받지 않습니다.

먼저 문장을 토큰 단위로 나누고,  
각 토큰을 숫자 ID로 바꿔야 합니다.

이 과정을 토크나이징이라고 합니다.

In [ ]:
def tokenize_function(examples):
    return tokenizer(
        examples["document"],
        padding="max_length",
        truncation=True,
        max_length=64
    )

tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

## 9. 학습에 필요한 컬럼만 남기기

모델 학습에는 토큰화된 입력값과 라벨만 필요합니다.

따라서 `id`, `document` 같은 원본 텍스트 컬럼은 제거하고,  
학습에 필요한 형태로 데이터셋을 정리합니다.

In [ ]:
# 현재 데이터셋에 실제로 어떤 컬럼이 있는지 먼저 확인
print(tokenized_train.column_names)
print(tokenized_test.column_names)

['id', 'document', 'label', 'input_ids', 'token_type_ids', 'attention_mask']
['id', 'document', 'label', 'input_ids', 'token_type_ids', 'attention_mask']


In [ ]:
# id 컬럼은 없을 수도 있으므로, 존재하는 컬럼만 제거
remove_cols_train = [col for col in ["id", "document"] if col in tokenized_train.column_names]
remove_cols_test = [col for col in ["id", "document"] if col in tokenized_test.column_names]

tokenized_train = tokenized_train.remove_columns(remove_cols_train)
tokenized_test = tokenized_test.remove_columns(remove_cols_test)

# set_format("torch")는 사용하지 않음
tokenized_train[0]

{'label': 1,
 'input_ids': [2,
  16919,
  16864,
  2010,
  18,
  1700,
  10223,
  2332,
  6233,
  3670,
  6396,
  1700,
  10223,
  2332,
  6233,
  21277,
  18,
  3,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0],
 'token_type_ids': [0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0],
 'attention_mask': [1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  

## 10. 파인튜닝할 모델 새로 만들기

이제 실제로 학습할 모델을 다시 만듭니다.

앞에서 만든 `model_before`는 파인튜닝 전 결과를 확인하기 위한 모델이었고,  
이번에는 학습에 사용할 모델을 새로 준비합니다.

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: klue/bert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## 11. 평가 지표 준비하기

감정분석은 분류 문제이므로 정확도 accuracy를 사용해 평가할 수 있습니다.

정확도는 전체 예측 중에서 정답을 맞힌 비율입니다.

In [ ]:
accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return accuracy.compute(predictions=predictions, references=labels)

## 12. 학습 설정하기

이제 학습 조건을 설정합니다.

처음 실습에서는 너무 오래 학습하지 않도록 `num_train_epochs=1`로 설정합니다.

시간이 충분하다면 2 또는 3으로 늘려볼 수 있습니다.

In [ ]:
training_args = TrainingArguments(
    output_dir="./nsmc_klue_bert_model",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=1,
    weight_decay=0.01,
    logging_steps=20,
    report_to="none",
    load_best_model_at_end=True
)

## 13. Trainer 만들기

Hugging Face의 `Trainer`를 사용하면 학습 루프를 직접 작성하지 않아도  
비교적 간단하게 모델을 학습시킬 수 있습니다.

`Trainer`는 PyTorch 기반 학습을 쉽게 실행할 수 있도록 도와주는 학습 API입니다. :contentReference[oaicite:2]{index=2}

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

## 14. 모델 파인튜닝하기

이제 모델을 학습합니다.

학습 데이터 2,000개만 사용하기 때문에 비교적 빠르게 끝납니다.

GPU를 사용하면 더 빠르게 학습할 수 있습니다.

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.393569,0.402193,0.837000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=188, training_loss=0.436916450236706, metrics={'train_runtime': 58.6605, 'train_samples_per_second': 51.142, 'train_steps_per_second': 3.205, 'total_flos': 98666645760000.0, 'train_loss': 0.436916450236706, 'epoch': 1.0})

## 15. 평가 결과 확인하기

학습이 끝난 뒤 테스트 데이터에 대해 정확도를 확인해봅니다.

데이터를 일부만 사용했기 때문에 아주 높은 성능을 기대하기보다는,  
파인튜닝 전보다 모델이 감정분석 작업을 수행할 수 있게 되었는지 확인하는 것이 중요합니다.

In [ ]:
trainer.evaluate()

Training Loss,Validation Loss,Epoch,Accuracy
0.393569,0.402193,1,0.837000


{'eval_loss': 0.402192622423172, 'eval_accuracy': 0.837}

## 16. 파인튜닝한 모델 저장하기

학습한 모델과 토크나이저를 저장합니다.

저장해두면 나중에 다시 불러와서 사용할 수 있습니다.

In [ ]:
trainer.save_model("./my_korean_sentiment_model")
tokenizer.save_pretrained("./my_korean_sentiment_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./my_korean_sentiment_model/tokenizer_config.json',
 './my_korean_sentiment_model/tokenizer.json')

## 17. 파인튜닝 후 예측하기

이제 학습이 끝난 모델을 사용해서 새로운 문장을 예측해봅시다.

파인튜닝 전 결과와 비교해보면,  
모델이 긍정/부정 분류를 더 잘 수행하게 되었는지 확인할 수 있습니다.

In [ ]:
after_classifier = pipeline(
    "text-classification",
    model="./my_korean_sentiment_model",
    tokenizer="./my_korean_sentiment_model",
    device=0 if torch.cuda.is_available() else -1
)

for text in test_texts:
    result = after_classifier(text)
    print("입력:", text)
    print("파인튜닝 후 결과:", result)
    print()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

입력: 정말 재미있고 감동적인 영화였다.
파인튜닝 후 결과: [{'label': 'LABEL_1', 'score': 0.9808949828147888}]

입력: 시간이 아까울 정도로 지루했다.
파인튜닝 후 결과: [{'label': 'LABEL_0', 'score': 0.8832976222038269}]

입력: 배우들의 연기가 훌륭했다.
파인튜닝 후 결과: [{'label': 'LABEL_1', 'score': 0.9638364911079407}]

입력: 스토리가 너무 엉성하고 재미없었다.
파인튜닝 후 결과: [{'label': 'LABEL_0', 'score': 0.9370212554931641}]



## 18. 라벨 이름을 보기 좋게 바꾸기

현재 결과는 `LABEL_0`, `LABEL_1`처럼 표시될 수 있습니다.

NSMC에서는 다음과 같이 해석하면 됩니다.

```text
LABEL_0 = 부정
LABEL_1 = 긍정

In [ ]:
label_map = {
    "LABEL_0": "부정",
    "LABEL_1": "긍정"
}

def predict_sentiment(text):
    result = after_classifier(text)[0]
    label = label_map[result["label"]]
    score = result["score"]

    print("입력 문장:", text)
    print("예측 결과:", label)
    print("확신도:", round(score, 4))
    print()

predict_sentiment("정말 재미있고 감동적인 영화였다.")
predict_sentiment("배우 연기는 좋았지만 스토리가 너무 지루했다.")
predict_sentiment("돈 주고 보기에는 너무 아까운 영화였다.")
predict_sentiment("기대보다 훨씬 재미있었다.")

입력 문장: 정말 재미있고 감동적인 영화였다.
예측 결과: 긍정
확신도: 0.9809

입력 문장: 배우 연기는 좋았지만 스토리가 너무 지루했다.
예측 결과: 부정
확신도: 0.8849

입력 문장: 돈 주고 보기에는 너무 아까운 영화였다.
예측 결과: 부정
확신도: 0.6992

입력 문장: 기대보다 훨씬 재미있었다.
예측 결과: 긍정
확신도: 0.9572



my_texts = [
    "이 영화는 생각보다 훨씬 재미있었다.",
    "내용이 너무 뻔하고 지루했다.",
    "배우는 좋았지만 결말은 아쉬웠다.",
    "OST와 영상미가 정말 훌륭했다.",
    "다시는 보고 싶지 않은 영화였다."
]

for text in my_texts:
    predict_sentiment(text)

In [ ]:
my_texts = [
    "이 영화는 생각보다 훨씬 재미있었다.",
    "내용이 너무 뻔하고 지루했다.",
    "배우는 좋았지만 결말은 아쉬웠다.",
    "OST와 영상미가 정말 훌륭했다.",
    "다시는 보고 싶지 않은 영화였다."
]

for text in my_texts:
    predict_sentiment(text)

입력 문장: 이 영화는 생각보다 훨씬 재미있었다.
예측 결과: 긍정
확신도: 0.9638

입력 문장: 내용이 너무 뻔하고 지루했다.
예측 결과: 부정
확신도: 0.9266

입력 문장: 배우는 좋았지만 결말은 아쉬웠다.
예측 결과: 부정
확신도: 0.5984

입력 문장: OST와 영상미가 정말 훌륭했다.
예측 결과: 긍정
확신도: 0.9757

입력 문장: 다시는 보고 싶지 않은 영화였다.
예측 결과: 긍정
확신도: 0.8175



## 20. 실습 미션

아래 내용을 직접 바꿔보면서 결과를 비교해봅시다.

1. 학습 데이터 개수를 바꿔보세요.
   - 예: 1000개, 3000개, 5000개

2. 학습 횟수 `num_train_epochs`를 바꿔보세요.
   - 예: 1, 2, 3

3. 입력 문장의 길이를 바꿔보세요.
   - 짧은 리뷰
   - 긴 리뷰
   - 애매한 리뷰

4. 파인튜닝 전 결과와 파인튜닝 후 결과를 비교해보세요.

5. 모델을 `beomi/kcbert-base`로 바꾸면 결과가 어떻게 달라지는지 확인해보세요.